# E051 — Stress test: E033 image + E037/E042 audio

Compares new flagships against E028 results under same conditions.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import copy
import io

import librosa
import numpy as np
from PIL import Image
from scipy.ndimage import rotate as nd_rotate
from scipy.ndimage import gaussian_filter
from scipy.special import logsumexp
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer

SEED = 67
DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
print(f'{len(manifest)} samples loaded')

222 samples loaded


## Image helpers

In [2]:
def _find_png(stem, data_dir):
    for sf in ('target_train','target_dev','non_target_train','non_target_dev'):
        p = data_dir / sf / (stem + '.png')
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(path):
    img = Image.open(path).convert('RGB')
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2).flatten()

# --- augmentations (train only) ---
def _aug_flip(x): return x.reshape(80,80)[:,::-1].flatten()
def _aug_brightness(x, rng): return np.clip(x * rng.uniform(0.7,1.3), 0, 255)
def _aug_noise(x, rng): return np.clip(x + rng.normal(0,15,x.shape), 0, 255)
def _aug_rotate(x, angle):
    return nd_rotate(x.reshape(80,80), angle, reshape=False,
                     order=1, mode='constant', cval=0).flatten()

def _find_adv_rotation(x, scaler, pca, clf, max_angle=10.0, n_steps=5):
    best_angle, worst_abs = 0.0, np.inf
    for angle in np.linspace(-max_angle, max_angle, n_steps):
        if abs(angle) < 0.1: continue
        logit = clf.decision_function(
            pca.transform(scaler.transform(_aug_rotate(x, angle).reshape(1,-1)))
        )[0]
        if abs(logit) < worst_abs:
            worst_abs, best_angle = abs(logit), angle
    return best_angle

# --- val-time stresses ---
def _stress_jpeg(x, quality=15):
    img = Image.fromarray(x.reshape(80,80).astype(np.uint8))
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf), dtype=np.float32).flatten()

def _stress_blur(x, sigma=3.0):
    return gaussian_filter(x.reshape(80,80), sigma=sigma).flatten()

def _stress_rotate(x, angle=15.0):
    return nd_rotate(x.reshape(80,80), angle, reshape=False,
                     order=1, mode='constant', cval=0).flatten()

def _stress_occlude(x, size=20, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    img = x.reshape(80,80).copy()
    r = rng.integers(0, 80-size)
    c = rng.integers(0, 80-size)
    img[r:r+size, c:c+size] = 0
    return img.flatten()

def _stress_downsample(x):
    img = Image.fromarray(x.reshape(80,80).astype(np.uint8))
    img = img.resize((40,40), Image.BILINEAR).resize((80,80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).flatten()

def _apply_stress(x, stress_name):
    rng = np.random.default_rng(99)
    if stress_name == 'clean':        return x
    elif stress_name == 'jpeg':       return _stress_jpeg(x)
    elif stress_name == 'blur':       return _stress_blur(x)
    elif stress_name == 'rot15':      return _stress_rotate(x, 15)
    elif stress_name == 'rot25':      return _stress_rotate(x, 25)
    elif stress_name == 'occlude':    return _stress_occlude(x, rng=rng)
    elif stress_name == 'downsample': return _stress_downsample(x)
    elif stress_name == 'all':
        x = _stress_jpeg(x)
        x = _stress_blur(x)
        x = _stress_rotate(x, 15)
        x = _stress_occlude(x, rng=rng)
        return x
    raise ValueError(stress_name)

print('Image helpers ready')

Image helpers ready


## Image model training

In [3]:
N_PCA, C_LR = 50, 1.0

def _train_e007(df, data_dir, seed):
    """E007: +All aug (flip + brightness + noise), single pass."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        for v in [orig, _aug_flip(orig), _aug_brightness(orig,rng), _aug_noise(orig,rng)]:
            X.append(v); y.append(row['label'])
    X = np.stack(X); y = np.array(y)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X)), y)
    return scaler, pca, clf

def _train_e033(df, data_dir, seed):
    """E033: +All aug + adversarial rotation (2-pass)."""
    rng = np.random.default_rng(seed)
    X_basic, y_basic = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        for v in [orig, _aug_flip(orig), _aug_brightness(orig,rng), _aug_noise(orig,rng)]:
            X_basic.append(v); y_basic.append(row['label'])
    X_basic = np.stack(X_basic); y_basic = np.array(y_basic)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_basic)), y_basic)

    X_adv, y_adv = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        angle = _find_adv_rotation(orig, scaler, pca, clf)
        X_adv.append(_aug_rotate(orig, angle)); y_adv.append(row['label'])

    X_all = np.vstack([X_basic, np.stack(X_adv)])
    y_all = np.concatenate([y_basic, np.array(y_adv)])
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_all)), y_all)
    return scaler, pca, clf

def _score_image(x_stressed, scaler, pca, clf):
    return float(clf.decision_function(pca.transform(scaler.transform(x_stressed.reshape(1,-1))))[0])

print('Image train functions ready')

Image train functions ready


## Run image stress test

In [4]:
STRESSES = ['clean','jpeg','blur','downsample','rot15','rot25','occlude','all']
MODELS = {'E007 (+All)': _train_e007, 'E033 (+AdvRot)': _train_e033}

img_results = {name: {s: [] for s in STRESSES} for name in MODELS}

for model_name, train_fn in MODELS.items():
    print(f'\n=== {model_name} ===')
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        print(f'  fold {fold_id}...', end=' ', flush=True)
        scaler, pca, clf = train_fn(manifest.loc[train_idx], DATA, SEED + fold_id)
        val_df = manifest.loc[val_idx]

        for stress in STRESSES:
            scores, labels = [], []
            for _, row in val_df.iterrows():
                x = _load_image(_find_png(row['stem'], DATA))
                x_s = _apply_stress(x, stress)
                scores.append(_score_image(x_s, scaler, pca, clf))
                labels.append(row['label'])
            eer, _ = compute_eer(np.array(scores)[np.array(labels)==1],
                                 np.array(scores)[np.array(labels)==0])
            img_results[model_name][stress].append(eer * 100)
        print('done')

print('\nImage stress test complete')


=== E007 (+All) ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

=== E033 (+AdvRot) ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

Image stress test complete


In [5]:
print('\n=== IMAGE STRESS TEST RESULTS ===\n')
header = f'{"Stress":<16}' + ''.join(f'{n:<22}' for n in MODELS) + 'Delta (E033-E007)'
print(header)
print('-' * len(header))

e028_baseline = {
    'clean': 0.97, 'jpeg': 1.25, 'blur': 1.71, 'downsample': 0.97,
    'rot15': 7.31, 'rot25': 13.56, 'occlude': 18.10, 'all': 26.16
}

for stress in STRESSES:
    row_vals = {}
    for model_name in MODELS:
        folds = img_results[model_name][stress]
        row_vals[model_name] = (np.mean(folds), np.std(folds))
    e007_mean = row_vals['E007 (+All)'][0]
    e033_mean = row_vals['E033 (+AdvRot)'][0]
    delta = e033_mean - e007_mean
    sign = '+' if delta >= 0 else ''
    cells = [f'{stress:<16}']
    for model_name in MODELS:
        m, s = row_vals[model_name]
        cells.append(f'{m:.2f} ± {s:.2f}%{" ":<8}')
    cells.append(f'{sign}{delta:.2f}pp')
    print(''.join(cells))


=== IMAGE STRESS TEST RESULTS ===

Stress          E007 (+All)           E033 (+AdvRot)        Delta (E033-E007)
-----------------------------------------------------------------------------
clean           1.53 ± 0.52%        0.51 ± 0.36%        -1.02pp
jpeg            1.53 ± 0.52%        0.51 ± 0.36%        -1.02pp
blur            3.66 ± 3.06%        0.51 ± 0.36%        -3.15pp
downsample      1.25 ± 0.90%        0.51 ± 0.36%        -0.74pp
rot15           19.03 ± 12.42%        7.59 ± 5.39%        -11.44pp
rot25           37.18 ± 21.78%        29.95 ± 16.38%        -7.22pp
occlude         10.14 ± 4.62%        11.06 ± 6.76%        +0.93pp
all             25.60 ± 14.61%        15.19 ± 9.69%        -10.42pp


## Audio helpers

In [6]:
UBM_COMPONENTS, MAP_R = 32, 16.0

def _find_wav(stem, data_dir):
    for sf in ('target_train','target_dev','non_target_train','non_target_dev'):
        p = data_dir / sf / (stem + '.wav')
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            log_H = -np.log(np.abs(np.fft.rfft(a, n=512)) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _train_ubm(X): return GaussianMixture(n_components=UBM_COMPONENTS,
    covariance_type='tied', max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:,None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:,None]*mu_hat + (1-alpha[:,None])*ubm.means_
    return adapted

def _train_e042(df, data_dir, seed):
    """E042: LPCC + tied cov + pitch aug."""
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row['stem'], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row['label'] == 1:  # pitch aug only for target
            wavs.append(librosa.effects.pitch_shift(y_wav, sr=sr,
                        n_steps=float(rng.choice([-2,-1,1,2]))))
        else:  # non-target also gets pitch aug per E025 setup
            wavs.append(librosa.effects.pitch_shift(y_wav, sr=sr,
                        n_steps=float(rng.choice([-2,-1,1,2]))))
        for y_aug in wavs:
            f = _extract_lpcc(y_aug, sr)
            X_list.append(f); y_list.extend([row['label']] * len(f))
    X = np.vstack(X_list); y = np.array(y_list)
    ubm = _train_ubm(X[y==0])
    adapted = _map_adapt(ubm, X[y==1])
    return ubm, adapted

def _score_audio_tta(y_orig, sr, adapted, ubm):
    """Speed TTA: average LLR over 0.9x, 1.0x, 1.1x."""
    scores = []
    for rate in [0.9, 1.0, 1.1]:
        y = librosa.effects.time_stretch(y_orig, rate=rate) if rate != 1.0 else y_orig
        f = _extract_lpcc(y, sr)
        scores.append(float((adapted.score_samples(f) - ubm.score_samples(f)).mean()))
    return float(np.mean(scores))

# --- audio stresses (applied to val audio only) ---
def _astress_noise(y, snr_db):
    p = np.mean(y**2) + 1e-10
    noise = np.random.default_rng(42).normal(0, np.sqrt(p / 10**(snr_db/10)), len(y))
    return (y + noise.astype(y.dtype)).clip(-1, 1)

def _astress_speed(y, rate):
    return librosa.effects.time_stretch(y, rate=rate)

def _astress_codec(y, sr, bitrate='32k'):
    """Simulate codec via heavy MP3-like quantization (resample down/up)."""
    # Downsample to 8kHz and back — simulates phone/codec bandwidth loss
    y_down = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(y_down, orig_sr=8000, target_sr=sr)

def _apply_audio_stress(y, sr, stress_name):
    if stress_name == 'clean':     return y
    elif stress_name == 'noise20': return _astress_noise(y, 20)
    elif stress_name == 'noise10': return _astress_noise(y, 10)
    elif stress_name == 'noise5':  return _astress_noise(y, 5)
    elif stress_name == 'slow':    return _astress_speed(y, 0.8)
    elif stress_name == 'fast':    return _astress_speed(y, 1.2)
    elif stress_name == 'codec':   return _astress_codec(y, sr)
    elif stress_name == 'all':
        y = _astress_noise(y, 10)
        y = _astress_codec(y, sr)
        y = _astress_speed(y, 0.85)
        return y
    raise ValueError(stress_name)

print('Audio helpers ready')

Audio helpers ready


## Run audio stress test

In [7]:
AUDIO_STRESSES = ['clean','noise20','noise10','noise5','slow','fast','codec','all']
aud_results = {s: [] for s in AUDIO_STRESSES}

print('=== E042 (LPCC + tied cov + speed TTA) audio stress ===')
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    print(f'  fold {fold_id}...', end=' ', flush=True)
    ubm, adapted = _train_e042(manifest.loc[train_idx], DATA, SEED + fold_id)
    val_df = manifest.loc[val_idx]

    for stress in AUDIO_STRESSES:
        scores, labels = [], []
        for _, row in val_df.iterrows():
            y_orig, sr = librosa.load(_find_wav(row['stem'], DATA), sr=None, mono=True)
            y_s = _apply_audio_stress(y_orig, sr, stress)
            scores.append(_score_audio_tta(y_s, sr, adapted, ubm))
            labels.append(row['label'])
        eer, _ = compute_eer(np.array(scores)[np.array(labels)==1],
                             np.array(scores)[np.array(labels)==0])
        aud_results[stress].append(eer * 100)
    print('done')

print('\nAudio stress test complete')

=== E042 (LPCC + tied cov + speed TTA) audio stress ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

Audio stress test complete


In [8]:
print('\n=== AUDIO STRESS TEST RESULTS (E042) ===\n')
print(f'{"Stress":<14} {"F0":>6} {"F1":>6} {"F2":>6} {"Mean±Std":>16} {"Delta vs clean":>16}')
print('-' * 65)
clean_mean = np.mean(aud_results['clean'])
for stress in AUDIO_STRESSES:
    folds = aud_results[stress]
    m, s = np.mean(folds), np.std(folds)
    delta = m - clean_mean
    sign = '+' if delta >= 0 else ''
    print(f'{stress:<14} {folds[0]:>6.2f} {folds[1]:>6.2f} {folds[2]:>6.2f}'
          f' {m:>6.2f} ± {s:.2f}%  {sign}{delta:.2f}pp')


=== AUDIO STRESS TEST RESULTS (E042) ===

Stress             F0     F1     F2         Mean±Std   Delta vs clean
-----------------------------------------------------------------
clean            1.39   0.00   0.00   0.46 ± 0.65%  +0.00pp
noise20         10.56   0.83   1.67   4.35 ± 4.40%  +3.89pp
noise10         10.56   9.17   0.83   6.85 ± 4.29%  +6.39pp
noise5           3.47   4.17   3.33   3.66 ± 0.36%  +3.19pp
slow             1.39   0.00   0.83   0.74 ± 0.57%  +0.28pp
fast             0.69   0.00   0.00   0.23 ± 0.33%  -0.23pp
codec           18.33   9.17  12.50  13.33 ± 3.79%  +12.87pp
all              2.78  10.00  10.00   7.59 ± 3.40%  +7.13pp
